# Prediction and Submission Generation
Notebook này dùng model đã huấn luyện để tạo file submission.csv đúng định dạng mẫu.

Mục tiêu:
- Load model model_xgb_scale.joblib
- Đọc dữ liệu test từ data/test/test_feature_engineered.csv (ưu tiên) hoặc data/test/test_X.npz
- Dự đoán nhãn Depression
- Xuất file submission.csv theo format của sample_submission.csv

## Setup Paths
Cell này tự tìm thư mục gốc của project để notebook chạy ổn định dù mở từ thư mục nào.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from joblib import load
from scipy import sparse

In [ ]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw" / "sample_submission.csv").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy project root chứa data/raw/sample_submission.csv")

project_root = find_project_root(Path.cwd())
model_path = project_root / "artifacts" / "model_xgb_scale.joblib"
feature_pipeline_path = project_root / "artifacts" / "feature_pipeline.joblib"
sample_path = project_root / "data" / "raw" / "sample_submission.csv"
test_csv_path = project_root / "data" / "test" / "test_feature_engineered.csv"
test_npz_path = project_root / "data" / "test" / "test_X.npz"
output_path = project_root / "submission.csv"

print("Project root:", project_root)
print("Model path:", model_path)
print("Feature pipeline exists:", feature_pipeline_path.exists())
print("Test CSV exists:", test_csv_path.exists())
print("Test NPZ exists:", test_npz_path.exists())

Project root: D:\Downloads\HCMUS-HocTap\IDA\Lab\exploring-mental-health-data-lab
Model path: D:\Downloads\HCMUS-HocTap\IDA\Lab\exploring-mental-health-data-lab\artifacts\model_xgb_scale.joblib
Test CSV exists: True
Test NPZ exists: True


## Load Model and Prepare Test Features
Với model XGBoost, dữ liệu đầu vào phải là số (không chứa cột object).
Notebook sẽ ưu tiên dùng test_X.npz. Nếu chỉ dùng CSV có cột object, notebook sẽ cố gắng transform bằng feature_pipeline.joblib.

In [ ]:
model = load(model_path)
sample_submission = pd.read_csv(sample_path)

test_df = None
X_test = None
id_series = sample_submission["id"]
source_used = None

# Ưu tiên NPZ vì thường đã đúng format numeric mà model mong đợi.
if test_npz_path.exists():
    X_test = sparse.load_npz(test_npz_path)
    source_used = "data/test/test_X.npz"
elif test_csv_path.exists():
    test_df = pd.read_csv(test_csv_path)
    X_raw = test_df.drop(columns=["id"], errors="ignore")
    if "id" in test_df.columns:
        id_series = test_df["id"]

    object_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
    if object_cols:
        if feature_pipeline_path.exists():
            feature_pipeline = load(feature_pipeline_path)
            X_test = feature_pipeline.transform(X_raw)
            source_used = "data/test/test_feature_engineered.csv + artifacts/feature_pipeline.joblib"
            print(f"Transformed {len(object_cols)} object columns with feature pipeline.")
        else:
            raise ValueError(
                "Test CSV có cột object nhưng không có feature_pipeline.joblib để transform. "
                "Hãy dùng data/test/test_X.npz hoặc cung cấp pipeline."
            )
    else:
        X_test = X_raw
        source_used = "data/test/test_feature_engineered.csv (numeric-only)"
else:
    raise FileNotFoundError("Không tìm thấy cả test_feature_engineered.csv và test_X.npz")

print("Data source:", source_used)
if isinstance(X_test, pd.DataFrame):
    print("X_test shape:", X_test.shape)
else:
    print("X_test sparse shape:", X_test.shape)

if len(id_series) != X_test.shape[0]:
    raise ValueError(f"Số lượng id ({len(id_series)}) không khớp số dòng test ({X_test.shape[0]}).")

y_pred = model.predict(X_test)
if np.issubdtype(np.asarray(y_pred).dtype, np.floating):
    y_pred = (np.asarray(y_pred) >= 0.5).astype(int)
else:
    y_pred = np.asarray(y_pred).astype(int)

print("Predictions shape:", y_pred.shape)

Data source: ../data/test/test_feature_engineered.csv
X_test shape: (93800, 54)


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Gender: object, City: object, Working Professional or Student: object, Profession: object, Sleep Duration: object, Dietary Habits: object, Degree: object, Have you ever had suicidal thoughts ?: object, Family History of Mental Illness: object, age_group: object

## Build Submission File
Tạo DataFrame submission theo đúng định dạng mẫu: cột id và Depression.

In [ ]:
submission = pd.DataFrame({
    "id": id_series.values,
    "Depression": y_pred
})

submission = submission[sample_submission.columns.tolist()]
submission.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Submission shape:", submission.shape)
display(submission.head())